# Week 87, Day 2: Writing the specification

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w087d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('spec.md', 'w', encoding='utf-8') as f:
    f.write('# Waymark: specification\n\n## Problem\nPeople who walk for pleasure plan routes by tracing paths on paper maps or apps that only know roads. It is hard to find a walk of a chosen length from your door, and estimates ignore the climbing, so walks run late.\n\n## Users\n- Leisure walkers: walk for one to four hours at weekends; know their area; use a laptop to plan.\n- Walking group leaders: plan routes for groups of up to twenty, need times they can promise.\n\n## Stories\n- S1: As a walker, I want to find the shortest route between two places so that I can plan a walk without tracing paths by hand.\n- S2: As a walker, I want an estimated time that includes the climbing so that I don\'t run late.\n- S3: As a walker, I want to save a route with a name so that I can walk it again.\n- S4: As a group leader, I want a route close to a target time so that I can plan a morning walk.\n- S5: As a walker, I want to log walks I\'ve done so that I can see how far I\'ve walked this year.\n\n## Acceptance criteria\nScenario: S1 shortest route\n  Given the paths A-B (2 km), B-C (3 km) and A-C (6 km)\n  When I ask for a route from A to C\n  Then the route is A, B, C\n  And its length is 5 km\n\nScenario: S1 no route\n  Given places A and D with no paths joining them\n  When I ask for a route from A to D\n  Then I am told there is no route\n\nScenario: S2 time with climbing\n  Given a route of 10 km with 600 m of climbing\n  When I ask for its time\n  Then the estimate is 3 hours\n\nScenario: S3 saving a route\n  Given a route from A to C\n  When I save it as "Morning loop"\n  Then "Morning loop" appears in my saved routes\n\nScenario: S4 target time\n  Given a target of 2 hours from A\n  When I ask for a round walk\n  Then every route offered takes between 1 h 45 min and 2 h 15 min\n\n## Non-functional requirements\n- Route search answers within 200 ms for a map of 5,000 junctions on a laptop.\n- Times are estimated with Naismith\'s rule, to the nearest minute.\n- No data leaves the user\'s computer; the map and log are stored in a local database file.\n- The API returns JSON errors with a message for every 4xx response.\n\n## Out of scope\n- Turn-by-turn navigation and GPS tracking.\n- A phone app.\n- Accounts for more than one person.\n\n## Glossary\n- Junction: a place where paths meet, with a name and a height.\n- Path: a walkable link between two junctions, with a length and a climb in each direction.\n- Route: a sequence of junctions joined by paths.\n- Walk: a route someone has completed, with the date.\n')

**Predict before you run:** what will this print?



In [ ]:
import re

story = "As a walker, I want to save a route with a name so that I can walk it again."
m = re.match(r"As an? (.+?), I want (.+?) so that (.+?)\.?$", story)
print(m.group(1))
print(m.group(2))
print(m.group(3))

In [ ]:
import re

STORY = re.compile(r"As an? (?P<role>.+?), I want (?P<goal>.+?) so that (?P<benefit>.+?)\.?$")
for text in ["As a walker, I want to log walks I've done so that I can see how far I've walked this year.",
             "As a group leader, I want a route close to a target time so that I can plan a morning walk.",
             "The system shall implement route caching."]:
    m = STORY.match(text)
    print(m.groupdict() if m else f"not a story: {text!r}")

In [ ]:
spec = open("spec.md").read()
section = spec.split("## Acceptance criteria")[1].split("\n## ")[0]
scenarios = [block.strip() for block in section.split("Scenario:")[1:]]
for block in scenarios[:2]:
    title, *steps = block.splitlines()
    print(title)
    for step in steps:
        keyword = step.split()[0]
        print("   ", keyword.ljust(5), step.strip()[len(keyword):].strip())

> **Key idea.** A specification states the problem in the users' terms, describes users concretely, lists user stories ("As a ROLE, I want GOAL so that BENEFIT", each small, valuable, testable and free of design, with an id), gives each story acceptance criteria as Given/When/Then scenarios with real numbers and unhappy paths, sets measurable non-functional requirements (a number, a unit and a condition), lists what's out of scope, and defines its terms in a glossary. Keep it short, keep it in the repository, and change it in the same commit as the code. Scenarios become tests; stories trace to tests by id.

### Exercise: Parsing user stories

Write `parse_story(text)`: for a story like `"S3: As a walker, I want to save a route so that I can walk it again."` return a dict with `id` (`"S3"`, or `None` when there's no `ID: ` prefix), `role`, `goal` and `benefit` (without the final full stop). Accept "As a" or "As an", with or without the comma after the role. Raise `ValueError("not a user story")` for anything else.

In [ ]:
import re

def parse_story(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_story():\n    \"who, what and why\"\n    assert parse_story(\"S3: As a walker, I want to save a route with a name so that I can walk it again.\") == {\n        \"id\": \"S3\", \"role\": \"walker\", \"goal\": \"to save a route with a name\", \"benefit\": \"I can walk it again\"}\n    assert parse_story(\"As an organiser I want a list of walks so that I can plan the season\") == {\n        \"id\": None, \"role\": \"organiser\", \"goal\": \"a list of walks\", \"benefit\": \"I can plan the season\"}\n    for bad in (\"The system shall cache routes.\", \"As a walker, I want routes.\", \"\"):\n        try:\n            parse_story(bad)\n        except ValueError as e:\n            assert str(e) == \"not a user story\"\n        else:\n            raise AssertionError(bad)")

### Exercise: Is it a good story?

Write `story_problems(goal, benefit)` for a parsed story: a list (in this order, each included only if it applies) of `"two goals"` if the goal contains the word `and`; `"vague: W"` for each word in `VAGUE` (given) found in the goal or benefit, in the order of `VAGUE`; `"no real benefit"` if the benefit is empty or merely restates the goal (the benefit, lower-cased, contains the whole goal without its leading `"to "`); and `"design, not need"` if the goal mentions any word in `DESIGN` (given).

In [ ]:
import re

VAGUE = ["easy", "fast", "quickly", "user-friendly", "intuitive", "etc"]
DESIGN = ["endpoint", "database", "table", "class", "dijkstra", "json", "api"]

def story_problems(goal, benefit):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_checks():\n    \"small, specific, about needs\"\n    assert story_problems(\"to save a route with a name\", \"I can walk it again\") == []\n    assert story_problems(\"to find routes and save them\", \"I can plan walks\") == [\"two goals\"]\n    assert story_problems(\"an easy, fast way to log walks\", \"it is user-friendly\") == [\"vague: easy\", \"vague: fast\", \"vague: user-friendly\"]\n    assert story_problems(\"to export routes\", \"I can export routes\") == [\"no real benefit\"]\n    assert story_problems(\"a Dijkstra endpoint\", \"I can find routes\") == [\"design, not need\"]\n    assert story_problems(\"to see my walks\", \"\") == [\"no real benefit\"]")

### Exercise: Reading scenarios

Write `parse_scenarios(text)` for Given/When/Then text like the acceptance criteria in `spec.md`: return a list of dicts, one per `Scenario:` line, with `title` (the text after `Scenario:`, stripped), `story` (its first word if that looks like a story id such as `S1`, else `None`) and `given`, `when` and `then` (lists of step texts without their keyword). `And` adds to the same list as the step before it. Raise `ValueError("scenario T has no Then")` if a scenario has no Then step, and `ValueError("And with nothing before it in T")` for an `And` that's the first step. Ignore blank lines.

In [ ]:
import re

def parse_scenarios(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_scenarios():\n    \"Given, When, Then, And\"\n    spec = open(\"spec.md\").read()\n    section = spec.split(\"## Acceptance criteria\")[1].split(\"\\n## \")[0]\n    scenarios = parse_scenarios(section)\n    assert len(scenarios) == 5 and [s[\"story\"] for s in scenarios] == [\"S1\", \"S1\", \"S2\", \"S3\", \"S4\"]\n    first = scenarios[0]\n    assert first[\"title\"] == \"S1 shortest route\"\n    assert first[\"given\"] == [\"the paths A-B (2 km), B-C (3 km) and A-C (6 km)\"]\n    assert first[\"when\"] == [\"I ask for a route from A to C\"]\n    assert first[\"then\"] == [\"the route is A, B, C\", \"its length is 5 km\"]\n    assert parse_scenarios(\"Scenario: plain\\n  Given x\\n  And y\\n  When z\\n  Then w\\n\")[0] == {\n        \"title\": \"plain\", \"story\": None, \"given\": [\"x\", \"y\"], \"when\": [\"z\"], \"then\": [\"w\"]}\n    for text, message in ((\"Scenario: T1 broken\\n  Given x\\n  When y\\n\", \"scenario T1 broken has no Then\"),\n                          (\"Scenario: odd\\n  And x\\n  Then y\\n\", \"And with nothing before it in odd\")):\n        try:\n            parse_scenarios(text)\n        except ValueError as e:\n            assert str(e) == message, str(e)\n        else:\n            raise AssertionError(message)")

### Exercise: Measurable requirements

Write `measurable(requirement)`: `True` if the text contains a number (digits, possibly with commas or a decimal point) followed, possibly after a space, by a unit from `UNITS` (given), **and** a bound phrase from `BOUNDS`; ignore case. Then `unmeasurable(requirements)`: the requirements that aren't, in order.

In [ ]:
import re

UNITS = ["ms", "s", "seconds", "minutes", "%", "mb", "gb", "kb", "users", "requests", "junctions", "km"]
BOUNDS = ["within", "at most", "at least", "under", "below", "above", "no more than", "fewer than", "less than"]

def measurable(requirement):
    ___

def unmeasurable(requirements):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_nfr():\n    \"a number, a unit, a bound\"\n    assert measurable(\"Route search answers within 200 ms for a map of 5,000 junctions on a laptop.\")\n    assert measurable(\"The page loads in under 1.5 s on a slow connection\")\n    assert measurable(\"Supports at least 20 users at once\")\n    assert not measurable(\"Route search is fast.\")\n    assert not measurable(\"Uses 200 ms\")\n    assert not measurable(\"At most a few seconds\")\n    reqs = [\"It must be quick\", \"Errors within 2 seconds\", \"Under 5% of requests fail\", \"Looks nice\"]\n    assert unmeasurable(reqs) == [\"It must be quick\", \"Looks nice\"]")

### Exercise: Sections of a spec

Write `sections(markdown)`: a dict from each level-two heading (`## Name`) to the text under it (stripped), in order. Then `missing_sections(markdown)`: the names in `REQUIRED` (given) that are missing **or** have no text, in the order of `REQUIRED`.

In [ ]:
REQUIRED = ["Problem", "Users", "Stories", "Acceptance criteria", "Non-functional requirements", "Out of scope", "Glossary"]

def sections(markdown):
    ___

def missing_sections(markdown):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_sections():\n    \"headings and their text\"\n    spec = open(\"spec.md\").read()\n    s = sections(spec)\n    assert list(s) == [\"Problem\", \"Users\", \"Stories\", \"Acceptance criteria\", \"Non-functional requirements\", \"Out of scope\", \"Glossary\"]\n    assert s[\"Out of scope\"].splitlines()[0] == \"- Turn-by-turn navigation and GPS tracking.\"\n    assert missing_sections(spec) == []\n    thin = \"# X\\n\\n## Problem\\nIt's hard.\\n\\n## Users\\n\\n## Stories\\n- S1: As a a, I want b so that c\\n\"\n    assert missing_sections(thin) == [\"Users\", \"Acceptance criteria\", \"Non-functional requirements\", \"Out of scope\", \"Glossary\"]")

### Exercise: Stories to tests

Tests name the story they check: `test_s1_shortest_route` checks S1. Write `trace(story_ids, test_names)`: a dict from each story id (in the given order) to the sorted list of tests whose name starts with `test_` followed by the lower-cased id and `_`. Then `gaps(story_ids, test_names)`: `(untested, orphans)`: the story ids with no test, and the sorted test names that don't match any story.

In [ ]:
def trace(story_ids, test_names):
    ___

def gaps(story_ids, test_names):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_trace():\n    \"every story tested, every test explained\"\n    stories = [\"S1\", \"S2\", \"S3\", \"S12\"]\n    tests = [\"test_s1_no_route\", \"test_s1_shortest_route\", \"test_s2_time\", \"test_s12_export\", \"test_helpers\", \"test_s9_old\"]\n    assert trace(stories, tests) == {\"S1\": [\"test_s1_no_route\", \"test_s1_shortest_route\"], \"S2\": [\"test_s2_time\"], \"S3\": [], \"S12\": [\"test_s12_export\"]}\n    assert gaps(stories, tests) == ([\"S3\"], [\"test_helpers\", \"test_s9_old\"])")

### Bug hunt: A goal that swallowed the benefit

The pattern's middle part is **greedy**, so for a story with "so that" twice it takes everything up to the last one: it prints `to log walks so that I remember them`. Make it stop at the first "so that", so it prints `to log walks`.

In [ ]:
import re

story = "As a walker, I want to log walks so that I remember them so that I can see my year."
m = re.match(r"As an? (.+?), I want (.+) so that (.+)", story)
print(m.group(2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"lazy matching\"\n    expect_output(\"to log walks\")")

### Bug hunt: An And in the wrong place

`And` should continue the **previous** step's kind, but this always files it under Given: it prints `{'given': ['a route', 'its length is 5 km'], 'then': ['the route is A, B, C']}`. Make it print `{'given': ['a route'], 'then': ['the route is A, B, C', 'its length is 5 km']}`.

In [ ]:
steps = ["Given a route", "Then the route is A, B, C", "And its length is 5 km"]
parts = {"given": [], "then": []}
for step in steps:
    keyword, rest = step.split(" ", 1)
    if keyword == "And":
        parts["given"].append(rest)
    else:
        parts[keyword.lower()].append(rest)
print(parts)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"remember the last keyword\"\n    expect_output(\"{'given': ['a route'], 'then': ['the route is A, B, C', 'its length is 5 km']}\")")

### Bug hunt: Fast counts as a number

The check accepts any requirement containing a bound phrase, so "Searches are fast, at most instant" passes: it prints `[True, True]`. Require a number with a unit as well, so it prints `[True, False]`.

In [ ]:
import re

def measurable(text):
    return bool(re.search(r"\b(within|at most|under)\b", text.lower()))

print([measurable("Answers within 200 ms"), measurable("Searches are fast, at most instant")])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"number, unit and bound\"\n    expect_output(\"[True, False]\")")

### Puzzle: From story to test

Arrange the lines of a pytest test written straight from Waymark's first scenario: the Given sets up, the When acts, the Thens assert. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the role in "As an organiser, I want a calendar so that walks don't clash"
q = ___   # how many stories should "I want to find, save and share routes" become
r = ___   # True or False: "Search is quick" is a measurable requirement
s = ___   # the keyword that starts the step describing the action in a scenario
t = ___   # the spec section that answers "are we doing a phone app?"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"organiser\" and q == 3 and r is False, \"p, q and r\"\n    assert s == \"When\" and t == \"Out of scope\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a spec checker

Write `check_spec(markdown)` returning a report dict: `missing` (required sections missing or empty, as in `missing_sections`); `stories` (the ids of the lines in the Stories section that start with `- ` and parse as stories, in order); `bad_stories` (the text, without `- `, of list items in Stories that **don't** parse); `story_problems` (a dict from story id to its non-empty list of problems from `story_problems`); `scenarios` (a dict from story id to how many scenarios name it); `untested` (story ids with no scenario); `unknown` (sorted scenario story ids that aren't stories); `unmeasurable` (list items in Non-functional requirements that fail `measurable`, without `- `); and `ok`: `True` only if `missing`, `bad_stories`, `untested`, `unknown` and `unmeasurable` are all empty. The helpers from the exercises are given.

In [ ]:
import re

REQUIRED = ["Problem", "Users", "Stories", "Acceptance criteria", "Non-functional requirements", "Out of scope", "Glossary"]
VAGUE = ["easy", "fast", "quickly", "user-friendly", "intuitive", "etc"]
DESIGN = ["endpoint", "database", "table", "class", "dijkstra", "json", "api"]
UNITS = ["ms", "s", "seconds", "minutes", "%", "mb", "gb", "kb", "users", "requests", "junctions", "km"]
BOUNDS = ["within", "at most", "at least", "under", "below", "above", "no more than", "fewer than", "less than"]
STORY = re.compile(r"(?:(?P<id>[A-Z]+\d+): )?As an? (?P<role>.+?),? I want (?P<goal>.+?) so that (?P<benefit>.+?)\.?$")

def parse_story(text):
    m = STORY.match(text.strip())
    if not m:
        raise ValueError("not a user story")
    return m.groupdict()

def story_problems(goal, benefit):
    words = lambda t: re.findall(r"[a-z0-9'-]+", t.lower())
    problems = ["two goals"] if "and" in words(goal) else []
    problems += [f"vague: {w}" for w in VAGUE if w in words(goal) + words(benefit)]
    if not benefit.strip() or goal.lower().removeprefix("to ").strip() in benefit.lower():
        problems.append("no real benefit")
    if any(w in words(goal) for w in DESIGN):
        problems.append("design, not need")
    return problems

def measurable(requirement):
    text = requirement.lower()
    unit = "|".join(re.escape(u) for u in sorted(UNITS, key=len, reverse=True))
    return bool(re.search(r"\d[\d,]*(?:\.\d+)?\s?(" + unit + r")(?![a-z])", text)) and \
        any(re.search(r"\b" + b + r"\b", text) for b in BOUNDS)

def sections(markdown):
    result, name, lines = {}, None, []
    for line in markdown.splitlines():
        if line.startswith("## "):
            if name is not None:
                result[name] = "\n".join(lines).strip()
            name, lines = line[3:].strip(), []
        elif name is not None:
            lines.append(line)
    if name is not None:
        result[name] = "\n".join(lines).strip()
    return result

def check_spec(markdown):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_model_spec():\n    \"Waymark's spec passes\"\n    r = check_spec(open(\"spec.md\").read())\n    assert r[\"missing\"] == [] and r[\"stories\"] == [\"S1\", \"S2\", \"S3\", \"S4\", \"S5\"] and r[\"bad_stories\"] == []\n    assert r[\"scenarios\"] == {\"S1\": 2, \"S2\": 1, \"S3\": 1, \"S4\": 1, \"S5\": 0}\n    assert r[\"untested\"] == [\"S5\"] and r[\"unknown\"] == [] and r[\"story_problems\"] == {}\n    assert r[\"unmeasurable\"] == [\"Times are estimated with Naismith's rule, to the nearest minute.\",\n                                 \"No data leaves the user's computer; the map and log are stored in a local database file.\",\n                                 \"The API returns JSON errors with a message for every 4xx response.\"]\n    assert r[\"ok\"] is False\n\ndef test_problems():\n    \"everything a reviewer would flag\"\n    text = \"\"\"## Problem\nHard to plan.\n\n## Users\nWalkers.\n\n## Stories\n- S1: As a walker, I want an easy way to find and save routes so that I can walk.\n- The system shall use SQLite.\n- S2: As a walker, I want to see my log so that I can see my log.\n\n## Acceptance criteria\nScenario: S1 finding\n  Given paths\n  When I search\n  Then I get a route\n\nScenario: S7 old\n  Given x\n  When y\n  Then z\n\n## Non-functional requirements\n- Answers within 100 ms for 50 junctions.\n\n## Out of scope\n- Phones.\n\"\"\"\n    r = check_spec(text)\n    assert r[\"missing\"] == [\"Glossary\"] and r[\"stories\"] == [\"S1\", \"S2\"] and r[\"bad_stories\"] == [\"The system shall use SQLite.\"]\n    assert r[\"story_problems\"] == {\"S1\": [\"two goals\", \"vague: easy\"], \"S2\": [\"no real benefit\"]}\n    assert r[\"untested\"] == [\"S2\"] and r[\"unknown\"] == [\"S7\"] and r[\"unmeasurable\"] == [] and r[\"ok\"] is False")

> **Is the checker right about Waymark?.** The checker says Waymark's spec isn't "ok": S5 has no scenario, and three non-functional requirements aren't measurable by its rules. One of those is fair (S5 needs a scenario: "Given walks of 5 km and 8 km logged this year, when I ask for my total, then it's 13 km"). The other three are judgements a script can't make: a privacy rule ("no data leaves the computer") is testable without a number. Tools like this are for finding things to **look at**, not for deciding. Which of its complaints would you act on in your own spec?

## Warm-up

A user story has three parts: **who** (a walker), **what** they want to do (save a route with a name) and **why** (to walk it again). The why matters most: it tells you what "done" means and what you could do instead if the obvious design turns out to be hard.

## Why write it down

Everything in a spec is something you'd otherwise decide by accident, in the middle of writing code, without remembering what you decided last week. Writing it first:

- **finds the gaps while they're cheap**: "what happens when there's no route?" costs a sentence now and a redesign later;
- **lets someone else check it**: a friend, a future user, your design review on Thursday;
- **defines done**: each story is finished when its acceptance criteria pass, and the capstone is finished when its musts are;
- **protects the scope**: the out-of-scope list is the answer to every tempting new idea in week four.

Keep it **short** (two or three pages for a capstone) and **alive**: when you change your mind, change the spec in the same commit as the code.

A spec has these parts: the **problem** (in the users' terms, not yours), the **users**, the **stories**, their **acceptance criteria**, the **non-functional requirements**, what's **out of scope**, and a **glossary**. Open `spec.md` above: Waymark's is a model to copy.

## Users and stories

Describe **users** concretely: what they're trying to do, how often, with what, and what they already know. "Leisure walkers who walk for one to four hours at weekends" helps you decide things; "users" doesn't.

A **user story** is one thing a user can do, written as **"As a ROLE, I want GOAL so that BENEFIT."** Good stories are:

- **small**: finished in a few hours to a couple of days (split anything bigger: "find routes **and** save them" is two stories);
- **valuable on their own**: each delivers something a user would notice;
- **testable**: you can say exactly how you'd check it;
- **free of design**: "I want to find a route", not "I want a Dijkstra endpoint".

Give each story an **id** (`S1`, `S2`, ...) so criteria, tests, commits and issues can refer to it.

## Acceptance criteria

Each story needs **acceptance criteria**: concrete examples that, if they all hold, mean the story is done. Write them as **scenarios** in the **Given / When / Then** form (from behaviour-driven development):

- **Given** the situation before (the data, the state);
- **When** the action;
- **Then** what must be true afterwards; **And** adds another step of the same kind as the one above it.

Use real numbers ("paths of 2 km and 3 km", "a route of 10 km with 600 m of climbing") and include the **unhappy paths**: no route, an empty log, a name that's already taken. Every scenario turns almost directly into a test, which is the point: on Day 5 you'll write Waymark's first tests from these.

## The rest of a spec

- **Non-functional requirements** describe how well, not what: speed, size, privacy, reliability, accessibility. They must be **measurable**: "fast" can't be checked; "answers within 200 ms for a map of 5,000 junctions on a laptop" can. Each needs a number, a unit and a condition.
- **Out of scope**: the things a reasonable person might expect that you are **not** doing ("no phone app", "one user only"). It's the most useful section in week four.
- **Glossary**: the words that mean something specific in your program ("junction", "route", "walk"). Use them consistently in the spec, the code and the interface; a class called `Walk` that sometimes means a route causes bugs.
- **Open questions**: anything you haven't decided yet, written down, with when you'll decide.

## Practice

## Quiz

## Daily challenge

## Your capstone today

On your own computer, write `docs/spec.md` for your capstone with the seven sections, using Waymark's as a pattern:

1. **Problem** in your users' words, three or four sentences.
2. **Users**: one or two kinds, described concretely.
3. **Stories**: every must and should from yesterday's scope, each as `- S1: As a ..., I want ... so that ...`. Split any with an "and".
4. **Acceptance criteria**: at least one scenario per must, with real numbers, and at least two unhappy paths.
5. **Non-functional requirements**: three to five, measurable where they can be.
6. **Out of scope**: everything you deferred yesterday, and anything a reader might expect.
7. **Glossary**: every word with a specific meaning.

Run today's `check_spec` over it (paste the function into a script) and fix what it finds, or decide deliberately not to.

## Before you go

Give your spec to someone else and ask them for three questions it doesn't answer. Add the answers (or add them to an "Open questions" section with a date). Tomorrow is **architecture and design review**: turning the spec into parts, boundaries and a data model, and having it checked.